## Imports

In [1]:
from __future__ import annotations

import json
from contextlib import nullcontext
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

from walmart_forecasting.data import (
    load_merged_data,
)
from walmart_forecasting.features import (
    add_basic_features,
    add_exact_lag_features,
)
from walmart_forecasting.paths import (
    PROJECT_ROOT,
    RAW_DATA_DIR,
    SUBMISSIONS_DIR,
)
from walmart_forecasting.tracking import (
    setup_mlflow,
)


pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

/home/xizusha/Documents/ML/walmart-store-sales-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
WANDB_ENTITY = (
    "lkhiz23-free-university-of-tbilisi-"
)

WANDB_PROJECT = (
    "walmart-store-sales-forecasting"
)

PATCHTST_REGISTRY_ARTIFACT = (
    "wandb-registry-walmart-models/"
    "PatchTST:best"
)

LIGHTGBM_EXPERIMENT_NAME = (
    "LightGBM_Training"
)

LIGHTGBM_MODEL_URI = (
    "models:/m-d85909fdb4934fbe9a97893601b86670"
)

LIGHTGBM_RUN_ID = (
    "4798f650278941f99d5e0e32a68fca3b"
)

LIGHTGBM_LAGS = (
    52,
)

LIGHTGBM_WEIGHT = 0.50
PATCHTST_WEIGHT = 0.50

FORECAST_HORIZON = 13
MIN_INPUT_COVERAGE = 0.80

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = (
    DEVICE.type == "cuda"
)

PATCHTST_DOWNLOAD_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "patchtst_registry"
)

PATCHTST_DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not np.isclose(
    LIGHTGBM_WEIGHT
    + PATCHTST_WEIGHT,
    1.0,
):
    raise ValueError(
        "Ensemble weights must sum to 1."
    )

print(
    "Device:",
    DEVICE,
)

print(
    "PatchTST artifact:",
    PATCHTST_REGISTRY_ARTIFACT,
)

print(
    "LightGBM model:",
    LIGHTGBM_MODEL_URI,
)

print(
    "Ensemble weights:",
    {
        "LightGBM": LIGHTGBM_WEIGHT,
        "PatchTST": PATCHTST_WEIGHT,
    },
)

Device: cuda
PatchTST artifact: wandb-registry-walmart-models/PatchTST:best
LightGBM model: models:/m-d85909fdb4934fbe9a97893601b86670
Ensemble weights: {'LightGBM': 0.5, 'PatchTST': 0.5}


## Load the frozen PatchTST checkpoint from W&B Registry

In [3]:
wandb_api = wandb.Api(
    overrides={
        "entity": WANDB_ENTITY,
    }
)

patchtst_artifact = (
    wandb_api.artifact(
        name=(
            PATCHTST_REGISTRY_ARTIFACT
        )
    )
)

patchtst_artifact_directory = Path(
    patchtst_artifact.download(
        root=str(
            PATCHTST_DOWNLOAD_DIR
        )
    )
)

checkpoint_candidates = list(
    patchtst_artifact_directory.rglob(
        "patchtst_final.pt"
    )
)

if len(checkpoint_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one "
        "patchtst_final.pt file, "
        f"found {len(checkpoint_candidates)} "
        f"inside "
        f"{patchtst_artifact_directory}."
    )

checkpoint_path = (
    checkpoint_candidates[0]
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

required_checkpoint_keys = {
    "model_state_dict",
    "model_configuration",
    "forecast_horizon",
    "final_epochs",
    "training_end_date",
}

missing_checkpoint_keys = (
    required_checkpoint_keys
    - set(checkpoint)
)

if missing_checkpoint_keys:
    raise KeyError(
        "Checkpoint is missing keys: "
        f"{sorted(missing_checkpoint_keys)}"
    )

print(
    "Downloaded PatchTST artifact:",
    patchtst_artifact.name,
)

print(
    "PatchTST artifact version:",
    patchtst_artifact.version,
)

print(
    "PatchTST training end:",
    checkpoint[
        "training_end_date"
    ],
)

print(
    "PatchTST final epochs:",
    checkpoint[
        "final_epochs"
    ],
)

checkpoint[
    "model_configuration"
]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/xizusha/.netrc.
wandb:   8 of 8 files downloaded.  


Downloaded PatchTST artifact: PatchTST:best
PatchTST artifact version: v0
PatchTST training end: 2012-01-27 00:00:00
PatchTST final epochs: 53


{'trial_id': 'optimization_lr_0002',
 'trial_name': 'learning_rate_0002',
 'capacity_profile': 'lower_learning_rate',
 'feature_set': 'target_only_context52_patch13_stride4_v1',
 'preprocessing': 'masked_panel_revin_v1',
 'input_length': 52,
 'patch_length': 13,
 'stride': 4,
 'd_model': 128,
 'n_heads': 4,
 'encoder_layers': 3,
 'd_ff': 256,
 'dropout': 0.1,
 'head_dropout': 0.1,
 'use_revin': True,
 'use_amp': True,
 'learning_rate': 0.0002,
 'weight_decay': 0.0001,
 'batch_size': 512,
 'max_epochs': 80,
 'patience': 12}

## PatchTST architecture

In [4]:
class PatchTSTModel(nn.Module):
    def __init__(
        self,
        input_length: int,
        horizon: int,
        patch_length: int,
        stride: int,
        d_model: int,
        n_heads: int,
        encoder_layers: int,
        d_ff: int,
        dropout: float,
        head_dropout: float,
        use_revin: bool,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(
                "d_model must be divisible "
                "by n_heads."
            )

        self.input_length = input_length
        self.horizon = horizon
        self.patch_length = patch_length
        self.stride = stride
        self.use_revin = use_revin

        self.number_of_patches = (
            (
                input_length
                + stride
                - patch_length
            )
            // stride
            + 1
        )

        self.patch_projection = nn.Linear(
            2 * patch_length,
            d_model,
        )

        self.position_embedding = (
            nn.Parameter(
                torch.zeros(
                    1,
                    self.number_of_patches,
                    d_model,
                )
            )
        )

        self.input_dropout = nn.Dropout(
            dropout
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_ff,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer=encoder_layer,
                num_layers=encoder_layers,
                norm=nn.LayerNorm(d_model),
                enable_nested_tensor=False,
            )
        )

        self.head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(head_dropout),
            nn.Linear(
                self.number_of_patches
                * d_model,
                horizon,
            ),
        )

        self._reset_parameters()

    def _reset_parameters(
        self,
    ) -> None:
        for module in self.modules():
            if isinstance(
                module,
                nn.Linear,
            ):
                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.LayerNorm,
            ):
                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

        for layer in self.encoder.layers:
            nn.init.xavier_uniform_(
                layer.self_attn
                .in_proj_weight
            )

            if (
                layer.self_attn
                .in_proj_bias
                is not None
            ):
                nn.init.zeros_(
                    layer.self_attn
                    .in_proj_bias
                )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

    def _normalize(
        self,
        values: torch.Tensor,
        mask: torch.Tensor,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
        torch.Tensor,
    ]:
        mask = mask.to(
            values.dtype
        )

        if not self.use_revin:
            mean = torch.zeros(
                (
                    values.shape[0],
                    1,
                ),
                dtype=values.dtype,
                device=values.device,
            )

            standard_deviation = (
                torch.ones(
                    (
                        values.shape[0],
                        1,
                    ),
                    dtype=values.dtype,
                    device=values.device,
                )
            )

            return (
                values * mask,
                mean,
                standard_deviation,
            )

        count = (
            mask.sum(
                dim=1,
                keepdim=True,
            )
            .clamp_min(1.0)
        )

        mean = (
            (values * mask)
            .sum(
                dim=1,
                keepdim=True,
            )
            / count
        )

        centered = (
            values - mean
        ) * mask

        variance = (
            centered.pow(2)
            .sum(
                dim=1,
                keepdim=True,
            )
            / count
        )

        standard_deviation = (
            variance + 1e-5
        ).sqrt()

        normalized = (
            centered
            / standard_deviation
        )

        return (
            normalized,
            mean,
            standard_deviation,
        )

    def forward(
        self,
        values: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        (
            normalized,
            mean,
            standard_deviation,
        ) = self._normalize(
            values=values,
            mask=mask,
        )

        padded_values = F.pad(
            normalized.unsqueeze(1),
            (
                0,
                self.stride,
            ),
            mode="replicate",
        ).squeeze(1)

        padded_mask = F.pad(
            mask.unsqueeze(1),
            (
                0,
                self.stride,
            ),
            mode="replicate",
        ).squeeze(1)

        value_patches = (
            padded_values.unfold(
                dimension=1,
                size=self.patch_length,
                step=self.stride,
            )
        )

        mask_patches = (
            padded_mask.unfold(
                dimension=1,
                size=self.patch_length,
                step=self.stride,
            )
        )

        patch_features = torch.cat(
            [
                value_patches,
                mask_patches,
            ],
            dim=-1,
        )

        tokens = self.patch_projection(
            patch_features
        )

        tokens = (
            tokens
            + self.position_embedding[
                :,
                :tokens.shape[1],
            ]
        )

        tokens = self.input_dropout(
            tokens
        )

        encoded = self.encoder(
            tokens
        )

        normalized_forecast = (
            self.head(encoded)
        )

        forecast = (
            normalized_forecast
            * standard_deviation
            + mean
        )

        return forecast


def build_model(
    configuration: dict,
) -> PatchTSTModel:
    return PatchTSTModel(
        input_length=configuration[
            "input_length"
        ],
        horizon=FORECAST_HORIZON,
        patch_length=configuration[
            "patch_length"
        ],
        stride=configuration[
            "stride"
        ],
        d_model=configuration[
            "d_model"
        ],
        n_heads=configuration[
            "n_heads"
        ],
        encoder_layers=configuration[
            "encoder_layers"
        ],
        d_ff=configuration[
            "d_ff"
        ],
        dropout=configuration[
            "dropout"
        ],
        head_dropout=configuration[
            "head_dropout"
        ],
        use_revin=configuration[
            "use_revin"
        ],
    )


def amp_context(
    configuration: dict,
):
    use_amp = (
        USE_AMP
        and bool(
            configuration.get(
                "use_amp",
                True,
            )
        )
    )

    if use_amp:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()

In [5]:
model_configuration = dict(
    checkpoint[
        "model_configuration"
    ]
)

checkpoint_horizon = int(
    checkpoint[
        "forecast_horizon"
    ]
)

if (
    checkpoint_horizon
    != FORECAST_HORIZON
):
    raise ValueError(
        "Checkpoint horizon does not "
        "match inference horizon."
    )

model = build_model(
    model_configuration
).to(DEVICE)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

parameter_count = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    "Registered PatchTST loaded."
)

print(
    "Model parameters:",
    f"{parameter_count:,}",
)

print(
    "Input length:",
    model_configuration[
        "input_length"
    ],
)

print(
    "Forecast horizon:",
    FORECAST_HORIZON,
)

Registered PatchTST loaded.
Model parameters: 420,877
Input length: 52
Forecast horizon: 13


## Load the frozen LightGBM pipeline from MLflow

In [7]:
setup_mlflow(
    experiment_name=(
        LIGHTGBM_EXPERIMENT_NAME
    )
)

lightgbm_pipeline = (
    mlflow.pyfunc.load_model(
        LIGHTGBM_MODEL_URI
    )
)

print(
    "Loaded LightGBM model:",
    LIGHTGBM_MODEL_URI,
)

print(
    "Source MLflow run:",
    LIGHTGBM_RUN_ID,
)

Accessing as lkhiz23

Initialized MLflow to track repo "lkhiz23/walmart-store-sales-forecasting"

Repository lkhiz23/walmart-store-sales-forecasting initialized!

Loaded LightGBM model: models:/m-d85909fdb4934fbe9a97893601b86670
Source MLflow run: 4798f650278941f99d5e0e32a68fca3b


## Load competition data

In [8]:
data = load_merged_data()

full_train = data.train.copy()
merged_test = data.test.copy()

sample_submission = (
    data.sample_submission.copy()
)

full_train["Date"] = pd.to_datetime(
    full_train["Date"]
)

merged_test["Date"] = pd.to_datetime(
    merged_test["Date"]
)

raw_test_path = (
    RAW_DATA_DIR
    / "test.csv"
)

if not raw_test_path.exists():
    raise FileNotFoundError(
        f"Raw test file was not found: "
        f"{raw_test_path}"
    )

raw_test = pd.read_csv(
    raw_test_path,
    parse_dates=["Date"],
)

required_train_columns = {
    "Store",
    "Dept",
    "Date",
    "Weekly_Sales",
    "IsHoliday",
}

required_test_columns = {
    "Store",
    "Dept",
    "Date",
    "IsHoliday",
}

missing_train_columns = (
    required_train_columns
    - set(full_train.columns)
)

missing_test_columns = (
    required_test_columns
    - set(raw_test.columns)
)

if missing_train_columns:
    raise ValueError(
        "Training data is missing: "
        f"{sorted(missing_train_columns)}"
    )

if missing_test_columns:
    raise ValueError(
        "Test data is missing: "
        f"{sorted(missing_test_columns)}"
    )

if full_train.duplicated(
    subset=[
        "Store",
        "Dept",
        "Date",
    ]
).any():
    raise ValueError(
        "Duplicate training rows found."
    )

if raw_test.duplicated(
    subset=[
        "Store",
        "Dept",
        "Date",
    ]
).any():
    raise ValueError(
        "Duplicate test rows found."
    )

test_dates = pd.DatetimeIndex(
    raw_test["Date"]
    .drop_duplicates()
    .sort_values()
)

if (
    len(test_dates)
    % FORECAST_HORIZON
    != 0
):
    raise ValueError(
        "Test horizon must be "
        "divisible by 13."
    )

expected_test_start = (
    full_train["Date"].max()
    + pd.Timedelta(days=7)
)

if (
    test_dates.min()
    != expected_test_start
):
    raise ValueError(
        "Test dates do not begin one "
        "week after training data. "
        f"Expected {expected_test_start}, "
        f"got {test_dates.min()}."
    )

print(
    "Training rows:",
    len(full_train),
)

print(
    "Available history end:",
    full_train["Date"].max(),
)

print(
    "PatchTST weight-training end:",
    checkpoint[
        "training_end_date"
    ],
)

print(
    "Test rows:",
    len(raw_test),
)

print(
    "Test dates:",
    len(test_dates),
)

print(
    "Test range:",
    test_dates.min(),
    "to",
    test_dates.max(),
)

Training rows: 421570
Available history end: 2012-10-26 00:00:00
PatchTST weight-training end: 2012-01-27 00:00:00
Test rows: 115064
Test dates: 39
Test range: 2012-11-02 00:00:00 to 2013-07-26 00:00:00


/tmp/ipykernel_82537/3636418054.py:111: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  + pd.Timedelta(days=7)


## LightGBM inference

In [9]:
def prepare_lightgbm_features(
    rows: pd.DataFrame,
    history: pd.DataFrame,
    lags: tuple[int, ...],
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    ordered_rows = (
        rows
        .sort_values(
            [
                "Date",
                "Store",
                "Dept",
            ]
        )
        .reset_index(drop=True)
        .copy()
    )

    ordered_rows[
        "_row_order"
    ] = np.arange(
        len(ordered_rows)
    )

    features = add_basic_features(
        ordered_rows
    )

    if lags:
        features = (
            add_exact_lag_features(
                rows=features,
                history=history,
                lags=lags,
            )
        )

        for lag in lags:
            lag_column = (
                f"lag_{lag}"
            )

            features[
                f"{lag_column}_missing"
            ] = (
                features[
                    lag_column
                ]
                .isna()
                .astype("int8")
            )

    features = (
        features
        .sort_values(
            "_row_order"
        )
        .reset_index(drop=True)
    )

    ordered_rows = (
        ordered_rows
        .sort_values(
            "_row_order"
        )
        .drop(
            columns="_row_order"
        )
        .reset_index(drop=True)
    )

    features = features.drop(
        columns=[
            "_row_order",
            "Weekly_Sales",
            "Date",
        ],
        errors="ignore",
    )

    return (
        ordered_rows,
        features,
    )


(
    lightgbm_test_rows,
    lightgbm_test_features,
) = prepare_lightgbm_features(
    rows=merged_test,
    history=full_train,
    lags=LIGHTGBM_LAGS,
)

lightgbm_test_predictions = (
    np.asarray(
        lightgbm_pipeline.predict(
            lightgbm_test_features
        ),
        dtype=float,
    )
    .reshape(-1)
)

if (
    len(lightgbm_test_predictions)
    != len(lightgbm_test_rows)
):
    raise ValueError(
        "LightGBM prediction row count "
        "does not match test rows."
    )

if not np.isfinite(
    lightgbm_test_predictions
).all():
    raise ValueError(
        "LightGBM predictions contain "
        "non-finite values."
    )

lightgbm_prediction_frame = (
    lightgbm_test_rows[
        [
            "Store",
            "Dept",
            "Date",
        ]
    ]
    .copy()
)

lightgbm_prediction_frame[
    "LightGBMPrediction"
] = lightgbm_test_predictions

print(
    "LightGBM predictions:",
    len(
        lightgbm_prediction_frame
    ),
)

lightgbm_prediction_frame[
    "LightGBMPrediction"
].describe()

/home/xizusha/Documents/ML/walmart-store-sales-forecasting/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM predictions: 115064


count   115,064.0000
mean     16,443.2093
std      23,320.9667
min      -4,299.3822
25%       2,113.8392
50%       7,445.4837
75%      20,841.3085
max     215,819.3955
Name: LightGBMPrediction, dtype: float64

## PatchTST inference panel

In [10]:
def build_inference_panel(
    training_rows: pd.DataFrame,
    forecast_rows: pd.DataFrame,
) -> dict:
    training_rows = (
        training_rows.copy()
    )

    forecast_rows = (
        forecast_rows.copy()
    )

    all_series = pd.concat(
        [
            training_rows[
                ["Store", "Dept"]
            ],
            forecast_rows[
                ["Store", "Dept"]
            ],
        ],
        ignore_index=True,
    )

    series_frame = (
        all_series
        .drop_duplicates()
        .sort_values(
            ["Store", "Dept"]
        )
        .reset_index(drop=True)
    )

    series_index = (
        pd.MultiIndex.from_frame(
            series_frame
        )
    )

    training_dates = pd.date_range(
        start=training_rows[
            "Date"
        ].min(),
        end=training_rows[
            "Date"
        ].max(),
        freq="W-FRI",
    )

    target_panel = (
        training_rows
        .pivot_table(
            index=[
                "Store",
                "Dept",
            ],
            columns="Date",
            values="Weekly_Sales",
            aggfunc="first",
        )
        .reindex(
            index=series_index,
            columns=training_dates,
        )
    )

    values = (
        target_panel
        .fillna(0.0)
        .to_numpy(
            dtype=np.float32
        )
    )

    observed = (
        target_panel
        .notna()
        .to_numpy(
            dtype=np.float32
        )
    )

    pair_means = (
        training_rows
        .groupby(
            ["Store", "Dept"]
        )["Weekly_Sales"]
        .mean()
    )

    department_means = (
        training_rows
        .groupby("Dept")[
            "Weekly_Sales"
        ]
        .mean()
    )

    store_means = (
        training_rows
        .groupby("Store")[
            "Weekly_Sales"
        ]
        .mean()
    )

    global_mean = float(
        training_rows[
            "Weekly_Sales"
        ].mean()
    )

    return {
        "series_frame": series_frame,
        "series_index": series_index,
        "dates": training_dates,
        "values": values,
        "observed": observed,
        "pair_means": pair_means,
        "department_means": (
            department_means
        ),
        "store_means": store_means,
        "global_mean": global_mean,
    }


inference_panel = build_inference_panel(
    training_rows=full_train,
    forecast_rows=raw_test,
)

print(
    "Panel series:",
    len(
        inference_panel[
            "series_frame"
        ]
    ),
)

print(
    "History dates:",
    len(
        inference_panel[
            "dates"
        ]
    ),
)

print(
    "History end:",
    inference_panel[
        "dates"
    ].max(),
)

Panel series: 3342
History dates: 143
History end: 2012-10-26 00:00:00


In [11]:
input_length = int(
    model_configuration[
        "input_length"
    ]
)

sample_values = torch.from_numpy(
    inference_panel[
        "values"
    ][
        :32,
        -input_length:,
    ]
).to(DEVICE)

sample_mask = torch.from_numpy(
    inference_panel[
        "observed"
    ][
        :32,
        -input_length:,
    ]
).to(DEVICE)

with torch.no_grad():
    with amp_context(
        model_configuration
    ):
        sample_predictions = model(
            sample_values,
            sample_mask,
        )

sample_predictions = (
    sample_predictions
    .float()
    .cpu()
    .numpy()
)

if not np.isfinite(
    sample_predictions
).all():
    raise FloatingPointError(
        "Loaded model produced "
        "non-finite predictions."
    )

print(
    "Model validation passed."
)

print(
    "Sample prediction shape:",
    sample_predictions.shape,
)

Model validation passed.
Sample prediction shape: (32, 13)


In [12]:
def make_fallback_predictions(
    rows: pd.DataFrame,
    panel_state: dict,
) -> np.ndarray:
    pair_keys = (
        pd.MultiIndex.from_frame(
            rows[
                ["Store", "Dept"]
            ]
        )
    )

    predictions = pd.Series(
        panel_state[
            "pair_means"
        ]
        .reindex(pair_keys)
        .to_numpy(),
        index=rows.index,
        dtype=float,
    )

    predictions = predictions.fillna(
        rows["Dept"].map(
            panel_state[
                "department_means"
            ]
        )
    )

    predictions = predictions.fillna(
        rows["Store"].map(
            panel_state[
                "store_means"
            ]
        )
    )

    predictions = predictions.fillna(
        panel_state[
            "global_mean"
        ]
    )

    return predictions.to_numpy(
        dtype=np.float32
    )

In [13]:
@torch.no_grad()
def recursive_predict(
    model: nn.Module,
    panel_state: dict,
    forecast_rows: pd.DataFrame,
    configuration: dict,
) -> pd.DataFrame:
    model.eval()

    forecast_rows = (
        forecast_rows.copy()
        .reset_index(drop=True)
    )

    forecast_rows[
        "_original_order"
    ] = np.arange(
        len(forecast_rows)
    )

    forecast_dates = (
        pd.DatetimeIndex(
            forecast_rows["Date"]
            .drop_duplicates()
            .sort_values()
        )
    )

    if (
        len(forecast_dates)
        % FORECAST_HORIZON
        != 0
    ):
        raise ValueError(
            "Forecast dates must be "
            "divisible into 13-week blocks."
        )

    input_length = int(
        configuration[
            "input_length"
        ]
    )

    values_history = (
        panel_state[
            "values"
        ].copy()
    )

    mask_history = (
        panel_state[
            "observed"
        ].copy()
    )

    latest_coverage = (
        mask_history[
            :,
            -input_length:,
        ]
        .mean(axis=1)
    )

    eligible = (
        latest_coverage
        >= MIN_INPUT_COVERAGE
    )

    eligible_indices = (
        np.flatnonzero(eligible)
    )

    prediction_frames = []

    batch_size = int(
        configuration[
            "batch_size"
        ]
    )

    for block_start in range(
        0,
        len(forecast_dates),
        FORECAST_HORIZON,
    ):
        block_dates = (
            forecast_dates[
                block_start:
                block_start
                + FORECAST_HORIZON
            ]
        )

        block_predictions = np.zeros(
            (
                len(
                    panel_state[
                        "series_frame"
                    ]
                ),
                FORECAST_HORIZON,
            ),
            dtype=np.float32,
        )

        for batch_start in range(
            0,
            len(eligible_indices),
            batch_size,
        ):
            batch_indices = (
                eligible_indices[
                    batch_start:
                    batch_start
                    + batch_size
                ]
            )

            batch_values = (
                torch.from_numpy(
                    values_history[
                        batch_indices,
                        -input_length:,
                    ]
                )
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            batch_mask = (
                torch.from_numpy(
                    mask_history[
                        batch_indices,
                        -input_length:,
                    ]
                )
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            with amp_context(
                configuration
            ):
                batch_predictions = model(
                    batch_values,
                    batch_mask,
                )

            batch_predictions = (
                batch_predictions
                .float()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            if not np.isfinite(
                batch_predictions
            ).all():
                raise FloatingPointError(
                    "Non-finite recursive "
                    "prediction detected."
                )

            block_predictions[
                batch_indices
            ] = batch_predictions

        eligible_series = (
            panel_state[
                "series_frame"
            ]
            .iloc[
                eligible_indices
            ]
            .reset_index(drop=True)
        )

        block_frame = (
            eligible_series
            .loc[
                eligible_series
                .index
                .repeat(
                    FORECAST_HORIZON
                )
            ]
            .reset_index(drop=True)
        )

        block_frame["Date"] = np.tile(
            block_dates.to_numpy(),
            len(eligible_series),
        )

        block_frame[
            "ModelPrediction"
        ] = (
            block_predictions[
                eligible_indices
            ]
            .reshape(-1)
        )

        prediction_frames.append(
            block_frame
        )

        predicted_mask = np.zeros_like(
            block_predictions,
            dtype=np.float32,
        )

        predicted_mask[
            eligible_indices
        ] = 1.0

        values_history = np.concatenate(
            [
                values_history,
                block_predictions,
            ],
            axis=1,
        )

        mask_history = np.concatenate(
            [
                mask_history,
                predicted_mask,
            ],
            axis=1,
        )

    neural_predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    result = forecast_rows.merge(
        neural_predictions,
        on=[
            "Store",
            "Dept",
            "Date",
        ],
        how="left",
        validate="many_to_one",
    )

    fallback_predictions = (
        make_fallback_predictions(
            rows=result,
            panel_state=panel_state,
        )
    )

    result[
        "UsedNeuralModel"
    ] = (
        result[
            "ModelPrediction"
        ].notna()
    )

    result["Prediction"] = np.where(
        result[
            "UsedNeuralModel"
        ],
        result[
            "ModelPrediction"
        ],
        fallback_predictions,
    )

    result = (
        result
        .sort_values(
            "_original_order"
        )
        .drop(
            columns=[
                "_original_order",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [14]:
patchtst_predictions = recursive_predict(
    model=model,
    panel_state=inference_panel,
    forecast_rows=raw_test,
    configuration=model_configuration,
)

if (
    len(patchtst_predictions)
    != len(raw_test)
):
    raise ValueError(
        "PatchTST prediction row count "
        "does not match test rows."
    )

if patchtst_predictions[
    "Prediction"
].isna().any():
    raise ValueError(
        "PatchTST predictions contain NaN."
    )

if not np.isfinite(
    patchtst_predictions[
        "Prediction"
    ].to_numpy()
).all():
    raise ValueError(
        "PatchTST predictions contain "
        "non-finite values."
    )

patchtst_coverage = float(
    patchtst_predictions[
        "UsedNeuralModel"
    ].mean()
)

patchtst_prediction_frame = (
    patchtst_predictions[
        [
            "Store",
            "Dept",
            "Date",
            "Prediction",
            "UsedNeuralModel",
        ]
    ]
    .rename(
        columns={
            "Prediction": (
                "PatchTSTPrediction"
            ),
        }
    )
)

print(
    "PatchTST predictions:",
    len(
        patchtst_prediction_frame
    ),
)

print(
    "PatchTST neural coverage:",
    f"{patchtst_coverage:.2%}",
)

patchtst_prediction_frame[
    "PatchTSTPrediction"
].describe()

PatchTST predictions: 115064
PatchTST neural coverage: 96.71%


count   115,064.0000
mean     16,389.8164
std      23,726.0000
min      -6,180.7471
25%       2,101.7628
50%       7,591.7937
75%      20,624.0674
max     637,966.0000
Name: PatchTSTPrediction, dtype: float64

## Align component predictions and create the equal-weight ensemble

In [15]:
KEY_COLUMNS = [
    "Store",
    "Dept",
    "Date",
]

ensemble_frame = (
    raw_test[
        KEY_COLUMNS
    ]
    .copy()
)

ensemble_frame[
    "_original_order"
] = np.arange(
    len(ensemble_frame)
)

ensemble_frame = (
    ensemble_frame
    .merge(
        lightgbm_prediction_frame,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    .merge(
        patchtst_prediction_frame,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "_original_order"
    )
    .reset_index(drop=True)
)

if ensemble_frame[
    [
        "LightGBMPrediction",
        "PatchTSTPrediction",
    ]
].isna().any().any():
    raise ValueError(
        "Component predictions did not "
        "align with all test rows."
    )

ensemble_frame[
    "EnsemblePrediction"
] = (
    LIGHTGBM_WEIGHT
    * ensemble_frame[
        "LightGBMPrediction"
    ]
    + PATCHTST_WEIGHT
    * ensemble_frame[
        "PatchTSTPrediction"
    ]
)

if not np.isfinite(
    ensemble_frame[
        "EnsemblePrediction"
    ].to_numpy()
).all():
    raise ValueError(
        "Ensemble predictions contain "
        "non-finite values."
    )

print(
    "Aligned prediction rows:",
    len(ensemble_frame),
)

ensemble_frame[
    [
        "LightGBMPrediction",
        "PatchTSTPrediction",
        "EnsemblePrediction",
    ]
].describe()

Aligned prediction rows: 115064


,LightGBMPrediction,PatchTSTPrediction,EnsemblePrediction
count,"115,064.0000","115,064.0000","115,064.0000"
mean,"16,443.2093","16,389.8164","16,416.5124"
std,"23,320.9667","23,726.0000","23,454.2549"
min,"-4,299.3822","-6,180.7471","-2,641.0883"
25%,"2,113.8392","2,101.7628","2,109.8162"
50%,"7,445.4837","7,591.7937","7,544.5105"
75%,"20,841.3085","20,624.0674","20,745.5959"
max,"215,819.3955","637,966.0000","420,001.0498"


## Build and validate Kaggle submissions

In [16]:
generated_ids = (
    raw_test["Store"]
    .astype(int)
    .astype(str)
    + "_"
    + raw_test["Dept"]
    .astype(int)
    .astype(str)
    + "_"
    + raw_test["Date"]
    .dt.strftime(
        "%Y-%m-%d"
    )
)

if (
    len(sample_submission)
    != len(ensemble_frame)
):
    raise ValueError(
        "Sample submission and "
        "prediction row counts differ."
    )

generated_id_series = (
    generated_ids
    .astype(str)
    .reset_index(drop=True)
)

sample_id_series = (
    sample_submission[
        "Id"
    ]
    .astype(str)
    .reset_index(drop=True)
)

if not generated_id_series.equals(
    sample_id_series
):
    raise ValueError(
        "Generated IDs do not match "
        "sampleSubmission.csv order."
    )

component_submission_frame = (
    ensemble_frame.copy()
)

component_submission_frame[
    "Id"
] = sample_submission[
    "Id"
].to_numpy()

lightgbm_submission = (
    component_submission_frame[
        [
            "Id",
            "LightGBMPrediction",
        ]
    ]
    .rename(
        columns={
            "LightGBMPrediction": (
                "Weekly_Sales"
            ),
        }
    )
)

patchtst_submission = (
    component_submission_frame[
        [
            "Id",
            "PatchTSTPrediction",
        ]
    ]
    .rename(
        columns={
            "PatchTSTPrediction": (
                "Weekly_Sales"
            ),
        }
    )
)

ensemble_submission = (
    component_submission_frame[
        [
            "Id",
            "EnsemblePrediction",
        ]
    ]
    .rename(
        columns={
            "EnsemblePrediction": (
                "Weekly_Sales"
            ),
        }
    )
)

for name, frame in {
    "LightGBM": lightgbm_submission,
    "PatchTST": patchtst_submission,
    "Ensemble": ensemble_submission,
}.items():
    if list(frame.columns) != [
        "Id",
        "Weekly_Sales",
    ]:
        raise ValueError(
            f"{name} submission has "
            "incorrect columns."
        )

    if len(frame) != len(raw_test):
        raise ValueError(
            f"{name} submission has "
            "incorrect row count."
        )

    if frame[
        "Id"
    ].duplicated().any():
        raise ValueError(
            f"{name} submission has "
            "duplicate IDs."
        )

    if frame[
        "Weekly_Sales"
    ].isna().any():
        raise ValueError(
            f"{name} submission contains "
            "missing predictions."
        )

    if not np.isfinite(
        frame[
            "Weekly_Sales"
        ].to_numpy()
    ).all():
        raise ValueError(
            f"{name} submission contains "
            "non-finite predictions."
        )

ensemble_submission.head(10)

,Id,Weekly_Sales
0,1_1_2012-11-02,"37,010.5753"
1,1_1_2012-11-09,"19,560.0839"
2,1_1_2012-11-16,"19,451.4669"
3,1_1_2012-11-23,"21,625.0203"
4,1_1_2012-11-30,"24,736.7300"
5,1_1_2012-12-07,"32,370.9971"
6,1_1_2012-12-14,"42,376.2877"
7,1_1_2012-12-21,"46,148.1737"
8,1_1_2012-12-28,"27,520.8309"
9,1_1_2013-01-04,"18,128.0924"


## Save component and ensemble submissions

In [17]:
lightgbm_submission_path = (
    SUBMISSIONS_DIR
    / "lightgbm_submission.csv"
)

patchtst_submission_path = (
    SUBMISSIONS_DIR
    / "patchtst_submission.csv"
)

ensemble_submission_path = (
    SUBMISSIONS_DIR
    / (
        "lightgbm_patchtst_"
        "equal_weight_submission.csv"
    )
)

lightgbm_submission.to_csv(
    lightgbm_submission_path,
    index=False,
)

patchtst_submission.to_csv(
    patchtst_submission_path,
    index=False,
)

ensemble_submission.to_csv(
    ensemble_submission_path,
    index=False,
)

manifest = {
    "ensemble_name": (
        "LightGBM-PatchTST "
        "Equal-Weight Ensemble"
    ),
    "forecast_strategy": (
        "heterogeneous_soft_voting_50_50"
    ),
    "weights": {
        "LightGBM": (
            LIGHTGBM_WEIGHT
        ),
        "PatchTST": (
            PATCHTST_WEIGHT
        ),
    },
    "lightgbm_model_uri": (
        LIGHTGBM_MODEL_URI
    ),
    "lightgbm_run_id": (
        LIGHTGBM_RUN_ID
    ),
    "patchtst_registry_artifact": (
        PATCHTST_REGISTRY_ARTIFACT
    ),
    "patchtst_artifact_version": (
        patchtst_artifact.version
    ),
    "patchtst_weight_training_end": str(
        checkpoint[
            "training_end_date"
        ]
    ),
    "available_input_history_end": str(
        full_train[
            "Date"
        ].max()
    ),
    "test_start": str(
        raw_test[
            "Date"
        ].min()
    ),
    "test_end": str(
        raw_test[
            "Date"
        ].max()
    ),
    "rows": len(
        ensemble_submission
    ),
    "patchtst_neural_coverage": (
        patchtst_coverage
    ),
    "retraining_performed": False,
}

manifest_path = (
    SUBMISSIONS_DIR
    / "ensemble_inference_manifest.json"
)

with manifest_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )

print(
    "LightGBM submission:",
    lightgbm_submission_path,
)

print(
    "PatchTST submission:",
    patchtst_submission_path,
)

print(
    "Final ensemble submission:",
    ensemble_submission_path,
)

print(
    "Inference manifest:",
    manifest_path,
)

print(
    "Rows:",
    len(
        ensemble_submission
    ),
)

print(
    "Final file size:",
    (
        f"{ensemble_submission_path.stat().st_size / 1024:.2f} KB"
    ),
)

LightGBM submission: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/lightgbm_submission.csv
PatchTST submission: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/patchtst_submission.csv
Final ensemble submission: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/lightgbm_patchtst_equal_weight_submission.csv
Inference manifest: /home/xizusha/Documents/ML/walmart-store-sales-forecasting/reports/submissions/ensemble_inference_manifest.json
Rows: 115064
Final file size: 3936.99 KB


## Prediction diagnostics

In [18]:
diagnostics = pd.DataFrame(
    [
        {
            "component": (
                "LightGBM"
            ),
            "source": (
                LIGHTGBM_MODEL_URI
            ),
            "weight": (
                LIGHTGBM_WEIGHT
            ),
            "minimum_prediction": float(
                ensemble_frame[
                    "LightGBMPrediction"
                ].min()
            ),
            "maximum_prediction": float(
                ensemble_frame[
                    "LightGBMPrediction"
                ].max()
            ),
            "mean_prediction": float(
                ensemble_frame[
                    "LightGBMPrediction"
                ].mean()
            ),
            "median_prediction": float(
                ensemble_frame[
                    "LightGBMPrediction"
                ].median()
            ),
        },
        {
            "component": (
                "PatchTST"
            ),
            "source": (
                PATCHTST_REGISTRY_ARTIFACT
            ),
            "weight": (
                PATCHTST_WEIGHT
            ),
            "minimum_prediction": float(
                ensemble_frame[
                    "PatchTSTPrediction"
                ].min()
            ),
            "maximum_prediction": float(
                ensemble_frame[
                    "PatchTSTPrediction"
                ].max()
            ),
            "mean_prediction": float(
                ensemble_frame[
                    "PatchTSTPrediction"
                ].mean()
            ),
            "median_prediction": float(
                ensemble_frame[
                    "PatchTSTPrediction"
                ].median()
            ),
        },
        {
            "component": (
                "Equal-weight ensemble"
            ),
            "source": (
                "LightGBM + PatchTST"
            ),
            "weight": 1.0,
            "minimum_prediction": float(
                ensemble_frame[
                    "EnsemblePrediction"
                ].min()
            ),
            "maximum_prediction": float(
                ensemble_frame[
                    "EnsemblePrediction"
                ].max()
            ),
            "mean_prediction": float(
                ensemble_frame[
                    "EnsemblePrediction"
                ].mean()
            ),
            "median_prediction": float(
                ensemble_frame[
                    "EnsemblePrediction"
                ].median()
            ),
        },
    ]
)

diagnostics

,component,source,weight,minimum_prediction,maximum_prediction,mean_prediction,median_prediction
0,LightGBM,models:/m-d85909fdb4934fbe9a97893601b86670,0.5000,"-4,299.3822","215,819.3955","16,443.2093","7,445.4837"
1,PatchTST,wandb-registry-walmart-models/PatchTST:best,0.5000,"-6,180.7471","637,966.0000","16,389.8164","7,591.7939"
2,Equal-weight ensemble,LightGBM + PatchTST,1.0000,"-2,641.0883","420,001.0498","16,416.5124","7,544.5105"
